In [20]:
import pandas as pd
import numpy as np
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean, RollingStd

# ==============================================================================
# PASSO 1: SIMULAÇÃO DE DADOS REAIS (ESTRUTURA WALMART M5)
# ==============================================================================
def generate_m5_simulated_data(n_stores=3, n_skus=5, start_date="2023-01-01", days=365, seed=42):
    """
    Gera um dataset simulando a dinâmica da competição M5 da Walmart:
    - Vendas de contagem (Poisson/Binomial Negativa) com zeros (intermitência).
    - Variações de preço com elasticidade-preço.
    - Sazonalidade semanal e eventos especiais.
    """
    np.random.seed(seed)
    dates = pd.date_range(start=start_date, periods=days, freq="D")
    n_days = len(dates)
    
    data_list = []

    for store_id in range(1, n_stores + 1):
        store_name = f"STORE_{store_id:02d}"
        for sku_id in range(1, n_skus + 1):
            item_id = f"FOODS_1_{sku_id:03d}"
            
            # Demanda base do SKU na loja
            base_demand = np.random.uniform(0.5, 5.0)
            
            # Fator de sazonalidade semanal (fds vende mais)
            dow_factor = np.tile([0.8, 0.85, 0.9, 0.95, 1.1, 1.4, 1.3], int(np.ceil(n_days/7)))[:n_days]
            
            # Variação e promoção de preço
            base_price = np.random.uniform(2.0, 15.0)
            prices = base_price * np.random.choice([1.0, 0.85, 0.70], size=n_days, p=[0.8, 0.15, 0.05])
            price_elasticity = np.exp(-0.15 * (prices - base_price))
            
            # Dias com eventos/promoções especiais
            event_indices = np.random.choice(n_days, size=12, replace=False)
            event_impact = np.ones(n_days)
            event_impact[event_indices] = np.random.uniform(1.3, 2.2, size=12)
            
            # Parâmetro lambda da demanda diária
            lambda_t = base_demand * dow_factor * price_elasticity * event_impact
            
            # Vendas reais observadas (intermitentes)
            sales = np.random.poisson(lambda_t)
            
            for d_idx, d_date in enumerate(dates):
                data_list.append({
                    'date': d_date,
                    'store_id': store_name,
                    'item_id': item_id,
                    'sales': sales[d_idx],
                    'sell_price': round(prices[d_idx], 2),
                    'is_event': 1 if d_idx in event_indices else 0,
                    'day_of_week': d_date.dayofweek, # 0=Segunda, 6=Domingo
                    'month': d_date.month,
                    'day_of_year': d_date.dayofyear
                })

    return pd.DataFrame(data_list)

df_raw = generate_m5_simulated_data(n_stores=3, n_skus=5, days=365) # Função criada anteriormente

df_raw['unique_id'] = df_raw['store_id'] + '_' + df_raw['item_id']

df_nixtla = df_raw.rename(columns={
    'date': 'ds',
    'sales': 'y'
})

In [21]:
# 2. Configurar o MLForecast com as classes nativas da Nixtla
fcst = MLForecast(
    models={}, # Dicionário vazio caso não queira treinar modelos agora
    freq='D',
    lags=[7, 14, 28], # Lags automáticos por unique_id
    lag_transforms={
        # Aplica média e desvio padrão móveis no lag 1 com janela de 7 dias
        1: [RollingMean(window_size=7), RollingStd(window_size=7)],
    },
    date_features=['dayofweek', 'month'] # Features de calendário automáticas
)

df_prepared_nixtla = fcst.preprocess(
    df_nixtla, 
    id_col='unique_id', 
    time_col='ds', 
    target_col='y',
    static_features=[]
)

In [34]:
import pandas as pd
import numpy as np
import holidays
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean, RollingStd
import pandas as pd
import holidays

# 1. Definir os feriados dos anos desejados
us_holidays = holidays.US(years=range(2022, 2027))

# 2. Pré-calcular o conjunto de datas da janela FORA da função (uma única vez na memória)
_extended_holiday_dates = set()
for h_date in us_holidays.keys():
    h_timestamp = pd.Timestamp(h_date)
    for offset in range(-2, 1):  # Véspera (-2, -1) e o próprio dia (0)
        _extended_holiday_dates.add((h_timestamp + pd.Timedelta(days=offset)).date())


# 3. Função limpa e ultra-rápida para o MLForecast
def is_holiday_window(dates) -> pd.Series:
    """Retorna 1 para o dia do feriado e os 2 dias que o antecedem."""
    dates_series = pd.Series(dates)
    return dates_series.dt.date.isin(_extended_holiday_dates).astype(int)

# Uso no MLForecast:
df_nixtla['is_holiday_window'] = is_holiday_window(df_nixtla['ds'])

# 3. Integrar diretamente ao MLForecast
fcst = MLForecast(
    models={},
    freq='D',
    lags=[7, 14, 28],
    lag_transforms={
        1: [RollingMean(window_size=7), RollingStd(window_size=7)],
    },
    # Passamos funções personalizadas diretamente na lista date_features!
    date_features=['dayofweek', 'month']
)

# 4. Processar o DataFrame
# O Nixtla aplicará a função 'is_holiday' automaticamente durante o preprocess
df_prepared_nixtla = fcst.preprocess(
    df_nixtla, 
    id_col='unique_id', 
    time_col='ds', 
    target_col='y',
    static_features=[]
)

In [35]:
df_prepared_nixtla

,ds,store_id,item_id,y,sell_price,is_event,day_of_week,month,day_of_year,unique_id,is_holiday_window,lag7,lag14,lag28,rolling_mean_lag1_window_size7,rolling_std_lag1_window_size7,dayofweek
28,2023-01-29,STORE_01,FOODS_1_001,0.0,14.36,0,6,1,29,STORE_01_FOODS_1_001,0,2.0,1.0,3.0,2.285714,1.496026,6
29,2023-01-30,STORE_01,FOODS_1_001,3.0,14.36,0,0,1,30,STORE_01_FOODS_1_001,0,3.0,0.0,3.0,2.000000,1.732051,0
30,2023-01-31,STORE_01,FOODS_1_001,3.0,14.36,0,1,1,31,STORE_01_FOODS_1_001,0,4.0,2.0,0.0,2.000000,1.732051,1
31,2023-02-01,STORE_01,FOODS_1_001,2.0,12.21,0,2,2,32,STORE_01_FOODS_1_001,0,2.0,1.0,1.0,1.857143,1.573592,2
32,2023-02-02,STORE_01,FOODS_1_001,7.0,10.05,0,3,2,33,STORE_01_FOODS_1_001,0,0.0,1.0,4.0,1.857143,1.573592,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5470,2023-12-27,STORE_03,FOODS_1_005,2.0,12.92,0,2,12,361,STORE_03_FOODS_1_005,0,7.0,10.0,2.0,4.428571,1.988058,2
5471,2023-12-28,STORE_03,FOODS_1_005,8.0,12.92,0,3,12,362,STORE_03_FOODS_1_005,0,3.0,6.0,4.0,3.714286,1.799469,3
5472,2023-12-29,STORE_03,FOODS_1_005,5.0,12.92,0,4,12,363,STORE_03_FOODS_1_005,0,5.0,6.0,10.0,4.428571,2.370452,4
5473,2023-12-30,STORE_03,FOODS_1_005,1.0,12.92,0,5,12,364,STORE_03_FOODS_1_005,1,3.0,0.0,7.0,4.428571,2.370452,5
